# Pixel refinement

**Data: Xinpu.** Sentinel-1A IW, ascending relative orbit 84, VV, 60 dates from 2019-11-26 to 2022-01-02 (reference 2020-12-02), cropped to the Xinpu landslide complex on the south bank of the Yangtze (Three Gorges Reservoir, 30.88-31.01N, 109.28-109.45E), 5130 x 1225 pixels (range x azimuth) at full resolution. Vegetated slopes with a few buildings: few PS, DS carry much of the signal. The data and how they were made are described in `data/Xinpu/README.md`.

The thresholds below are starting values; look at the figures and adjust them to the data.


In this tutorial, we demostrate how to merge previous processed PS candidates in [PS processing](./02_ps.ipynb) and DSs in [DS processing](./03_ds.ipynb) together and refine these measurement points with Noise2Fringe Transformer.

In [ ]:
import zarr
import numpy as np
from matplotlib import pyplot as plt
import colorcet
import moraine.cli as mc
import moraine as mr

## Merge pixels

In [ ]:
logger = mc.get_logger()

In [ ]:
ds_ph = './ds_processing/hix/ds/ds_ph.zarr'
ds_e = './ds_processing/hix/ds/ds_e.zarr'
ds_n = './ds_processing/hix/ds/ds_n.zarr'
ds_hix = './ds_processing/hix/ds/ds_hix.zarr'

ps_can_rslc = './ps_processing/hix/ps_can_rslc.zarr'
ps_can_e = './ps_processing/hix/ps_can_e.zarr'
ps_can_n = './ps_processing/hix/ps_can_n.zarr'
ps_can_hix = './ps_processing/hix/ps_can_hix.zarr'

pc_can_hix = './pixel_refinement/hix/pc_can_hix.zarr'
pc_can_ph = './pixel_refinement/hix/pc_can_ph.zarr'
pc_can_e = './pixel_refinement/hix/pc_can_e.zarr'
pc_can_n = './pixel_refinement/hix/pc_can_n.zarr'

In [ ]:
mc.pc_union(
    ps_can_hix, ds_hix, pc_can_hix,
    pc1 = [ps_can_rslc, ps_can_e, ps_can_n,],
    pc2 = [ds_ph, ds_e, ds_n],
    pc = [pc_can_ph, pc_can_e, pc_can_n],
)

In [ ]:
pc_can_lon = './pixel_refinement/hix/pc_can_lon.zarr'
pc_can_lat = './pixel_refinement/hix/pc_can_lat.zarr'

In [ ]:
mc.ras2pc(
    pc_can_hix, 
    ras=['./load_data/lon.zarr/','./load_data/lat.zarr/'],
    pc =[pc_can_lon, pc_can_lat]
)

## Refine pixels with Noise2fringe Transformer

In [ ]:
n2ft_intf = './pixel_refinement/hix/pc_can_n2f_intf.zarr'
pc_can_ph_zarr = zarr.open(pc_can_ph,mode='r')
tnet = tnet = mr.TempNet.from_bandwidth(pc_can_ph_zarr.shape[-1],1)

In [ ]:
mc.n2ft(pc_can_lon, pc_can_lat, pc_can_ph, n2ft_intf, tnet.image_pairs, chunks= 20000, cuda=True)

In [ ]:
temp_coh = './pixel_refinement/hix/pc_can_temp_coh.zarr'

In [ ]:
mc.temp_coh(n2ft_intf, pc_can_ph, temp_coh, tnet.image_pairs, cuda=True)

In [ ]:
mc.view(temp_coh, x=pc_can_e, y=pc_can_n, resolution=20, label='temporal coherence',
        cmap=colorcet.fire, clim=(0, 1))

In [ ]:
pc_hix = './pixel_refinement/hix/pc/pc_hix.zarr'

In [ ]:
t_coh_min = 0.8   # keep the points whose temporal coherence is above this
mc.pc_logic_pc(pc_can_hix, temp_coh, pc_hix, f'pc_in>{t_coh_min}')

In [ ]:
pc_ph = './pixel_refinement/hix/pc/pc_ph.zarr'
pc_e = './pixel_refinement/hix/pc/pc_e.zarr'
pc_n = './pixel_refinement/hix/pc/pc_n.zarr'

In [ ]:
mc.pc_select_data(
    pc_can_hix, pc_hix, 
    pc_in = [pc_can_ph, pc_can_e, pc_can_n],
    pc = [pc_ph, pc_e, pc_n],
)

## visualize the PS candidates

In [ ]:
pc_ph_pyramid = './pixel_refinement/hix/pc/pc_ph_pyramid'

In [ ]:
mc.pc_pyramid(
    pc_ph, pc_ph_pyramid,
    x = pc_e,
    y = pc_n,
    ras_resolution = 20,
)

In [ ]:
mc.view(pc_ph_pyramid, show='intf_all', dates='./load_data/meta.toml')